# Module 5, Topic 4 — Introduction to Vector Databases (ChromaDB, FAISS, pgvector, Pinecone)

**Generative AI Fellowship — Beginner**

In this notebook, we hand our search problem over to real vector database tools, instead of writing the brute-force loop ourselves like we did in Topic 3.

**What we'll do:**
1. Reuse a small embedded document set (fintech support queries + categories)
2. Set up a local **ChromaDB** collection and insert our documents
3. Run a similarity query against ChromaDB
4. Build the same vectors into a **FAISS** index and run the identical query
5. Compare what each tool returned
6. Walk through — conceptually, without full setup — how **pgvector** and **Pinecone** would handle the same task in production

> 💡 ChromaDB and FAISS both run locally with no account or server needed, so we can use them hands-on today. pgvector needs a running PostgreSQL instance, and Pinecone needs a cloud account — so for those two, we'll read and discuss code stubs instead of running them live.

## Step 1 — Install what we need

`chromadb` gives us a local, easy-to-use vector database. `faiss-cpu` gives us Meta AI's high-performance similarity search library. `gensim` lets us build embeddings the same way we did in Topics 2 and 3.

In [ ]:
!pip install chromadb faiss-cpu gensim numpy --quiet

## Step 2 — Import what we need

In [ ]:
import numpy as np
import chromadb
import faiss
from gensim.models import Word2Vec
from pprint import pprint

print("Ready to go!")

## Step 3 — Our document set

We're reusing the same kind of fintech customer-support queries from Topic 3 — this time, each one also has a **category**, which we'll store as metadata alongside its vector. Real vector databases are built to hold metadata like this next to every embedding, which is something our plain Python lists in Topic 3 couldn't do cleanly.

In [ ]:
documents = [
    {"id": "doc1",  "text": "how do i reset my transaction pin",         "category": "pin"},
    {"id": "doc2",  "text": "i forgot my pin how do i change it",        "category": "pin"},
    {"id": "doc3",  "text": "my transfer failed please help",            "category": "transfer"},
    {"id": "doc4",  "text": "why did my transfer fail",                  "category": "transfer"},
    {"id": "doc5",  "text": "how do i check my account balance",         "category": "balance"},
    {"id": "doc6",  "text": "i want to know my current balance",         "category": "balance"},
    {"id": "doc7",  "text": "how do i fund my wallet",                   "category": "wallet"},
    {"id": "doc8",  "text": "how can i add money to my wallet",          "category": "wallet"},
    {"id": "doc9",  "text": "my card was declined at the pos",           "category": "card"},
    {"id": "doc10", "text": "why was my card declined",                  "category": "card"},
    {"id": "doc11", "text": "how do i contact customer support",         "category": "support"},
    {"id": "doc12", "text": "i need to speak to an agent",               "category": "support"},
    {"id": "doc13", "text": "how do i verify my bvn",                    "category": "bvn"},
    {"id": "doc14", "text": "what is bvn verification for",              "category": "bvn"},
    {"id": "doc15", "text": "my app keeps crashing",                     "category": "app"},
    {"id": "doc16", "text": "the app is not opening on my phone",        "category": "app"},
]

print(f"{len(documents)} documents loaded, e.g.:")
pprint(documents[0])

## Step 4 — Build embeddings for each document

Same approach as Topic 3: train a small Word2Vec model on our corpus, then represent each document as the average of its word vectors.

In [ ]:
tokenized_docs = [doc["text"].split() for doc in documents]

word_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=30,
    window=4,
    min_count=1,
    sg=1,
    epochs=300,
    seed=42,
    workers=1,
)


def sentence_vector(sentence, model):
    words = sentence.split()
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    return np.mean(word_vectors, axis=0)


document_embeddings = [sentence_vector(doc["text"], word_model) for doc in documents]

print("Number of document embeddings:", len(document_embeddings))
print("Embedding shape:", document_embeddings[0].shape)

## Step 5 — Set up a local ChromaDB collection

A **collection** in ChromaDB is roughly like a table — a named group of vectors, their original text, and their metadata, stored together.

`chromadb.Client()` creates an in-memory database for this notebook session. In a real application, you'd typically use `chromadb.PersistentClient(path=...)` so the data survives after your program closes.

By default, ChromaDB ranks by squared Euclidean distance. Topic 3 showed us that Euclidean distance is sensitive to the magnitude noise in our averaged sentence vectors, while cosine similarity handles it correctly — so we explicitly configure this collection to use cosine distance instead, via the `hnsw:space` setting.

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="support_queries",
    metadata={"hnsw:space": "cosine"},  # use cosine distance, matching Topic 3's conclusion
)

print("Collection created:", collection.name)

## Step 6 — Insert our embedded documents

ChromaDB wants four parallel lists: the vector for each document, its original text, its metadata, and a unique id. Notice we don't write a search loop here — we're just handing our data to the database and letting it take over storage.

In [ ]:
collection.add(
    embeddings=[vector.tolist() for vector in document_embeddings],
    documents=[doc["text"] for doc in documents],
    metadatas=[{"category": doc["category"]} for doc in documents],
    ids=[doc["id"] for doc in documents],
)

print("Documents stored in ChromaDB:", collection.count())

## Step 7 — Run a similarity query against ChromaDB

Same query as Topic 3 — `"i cant remember my pin"` — but this time, ChromaDB does the searching, not our own loop.

In [ ]:
new_query = "i cant remember my pin"
new_query_vector = sentence_vector(new_query, word_model)

chroma_results = collection.query(
    query_embeddings=[new_query_vector.tolist()],
    n_results=5,
)

print(f'Query: "{new_query}"\n')
print("Top 5 matches (ChromaDB):")
for text, metadata, distance in zip(
    chroma_results["documents"][0],
    chroma_results["metadatas"][0],
    chroma_results["distances"][0],
):
    print(f"  [{metadata['category']:<9}] distance={distance:.3f}  -  {text}")

**What to notice:** we get back the matching text *and* its metadata (`category`) in one call — something our plain Python lists in Topic 3 couldn't give us without extra bookkeeping. Because we configured this collection for cosine distance, **smaller numbers mean closer matches** — and the top results should be the PIN-category documents, matching what cosine similarity found by hand in Topic 3.

## Step 8 — Build the same vectors into a FAISS index

FAISS is a library, not a full database — there's no metadata storage here, just raw vectors and their positions. We'll keep a separate Python list to map FAISS's numeric positions back to our document info.

`IndexFlatIP` ranks by inner product (dot product). Inner product on its own is magnitude-sensitive, just like Euclidean distance — but if we **normalise** every vector to length 1 first, inner product becomes mathematically equivalent to cosine similarity. We normalise here for the same reason we chose cosine distance for ChromaDB above: it's the metric that actually works for our averaged sentence vectors.

In [ ]:
def normalize(vector):
    return vector / np.linalg.norm(vector)


embedding_dimension = document_embeddings[0].shape[0]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

normalized_embeddings = np.array([normalize(v) for v in document_embeddings]).astype("float32")
faiss_index.add(normalized_embeddings)

print("Vectors stored in FAISS index:", faiss_index.ntotal)

## Step 9 — Run the identical query against FAISS

FAISS's `search` returns two arrays: the similarity scores, and the *positions* of the matching vectors — not the original text. This is the trade-off from the slides: FAISS is fast, but leaves storage and lookups to you.

Remember to normalise the query vector too — a normalised query against normalised stored vectors is what makes the inner-product score below equal to cosine similarity.

In [ ]:
query_vector_faiss = normalize(new_query_vector).astype("float32").reshape(1, -1)

top_k = 5
scores, positions = faiss_index.search(query_vector_faiss, top_k)

print(f'Query: "{new_query}"\n')
print("Top 5 matches (FAISS):")
for score, position in zip(scores[0], positions[0]):
    matched_doc = documents[position]
    print(f"  [{matched_doc['category']:<9}] score={score:.3f}  -  {matched_doc['text']}")

## Step 10 — ChromaDB vs. FAISS: comparing results

Both tools searched over the exact same 16 vectors and the exact same query, both configured to use cosine similarity/distance. Look at the two printed result lists above side by side.

**What to expect:** the top results from both tools should now largely agree, and should surface the PIN-category documents near the top — matching what we found by computing cosine similarity by hand in Topic 3. The remaining small differences in ordering come from ChromaDB and FAISS using slightly different underlying index implementations, not from a disagreement about which metric to use. That agreement is the real takeaway: **once you fix the metric, different vector database tools tend to agree on what "similar" means** — the metric choice matters far more than which specific tool you pick.

## Step 11 — How this would look in pgvector (conceptual walkthrough)

We won't run this cell — it requires a live PostgreSQL instance with the `pgvector` extension installed. Read through it as a conceptual comparison to Steps 5–7 above.

```sql
-- One-time setup: enable the extension
CREATE EXTENSION IF NOT EXISTS vector;

-- Create a table with a vector column (30 dimensions, matching our embeddings)
CREATE TABLE support_queries (
    id TEXT PRIMARY KEY,
    text TEXT,
    category TEXT,
    embedding VECTOR(30)
);

-- Insert a document — looks just like a normal SQL insert
INSERT INTO support_queries (id, text, category, embedding)
VALUES ('doc1', 'how do i reset my transaction pin', 'pin', '[0.12, -0.44, ...]');

-- Query for the 5 nearest neighbours using cosine distance (<=>)
SELECT text, category, embedding <=> '[0.11, -0.40, ...]' AS distance
FROM support_queries
ORDER BY distance
LIMIT 5;
```

**Compare this to Steps 5–7:** the shape of the workflow (create a table/collection → insert vectors with metadata → query for nearest neighbours) is identical to ChromaDB. The difference is that pgvector's version is plain SQL, running inside a database you might already have — you could even `JOIN` this query against a real `users` or `tickets` table in the same statement.

## Step 12 — How this would look in Pinecone (conceptual walkthrough)

We won't run this cell either — it requires a Pinecone account and API key. Read through it as a conceptual comparison.

```python
from pinecone import Pinecone

pc = Pinecone(api_key="your-api-key")

# One-time setup: create a hosted index
pc.create_index(name="support-queries", dimension=30, metric="cosine")
index = pc.Index("support-queries")

# Insert (upsert) documents — vector + metadata, just like ChromaDB
index.upsert(vectors=[
    {"id": "doc1", "values": [0.12, -0.44, "..."], "metadata": {"category": "pin", "text": "how do i reset my transaction pin"}},
])

# Query for the 5 nearest neighbours
results = index.query(vector=[0.11, -0.40, "..."], top_k=5, include_metadata=True)
```

**Compare this to Step 7 (ChromaDB):** the API shape is almost identical — create an index/collection, upsert vectors with metadata, query for top-k. The real difference isn't the code, it's *where it runs*: Pinecone's index lives on Pinecone's cloud servers, fully managed, rather than in this notebook's memory.

## Recap

In this notebook, we:
- Stored embedded documents, with metadata, in a local ChromaDB collection
- Queried ChromaDB for the nearest matches to a new question
- Built the same vectors into a FAISS index and ran the identical query
- Compared the two tools' results and score conventions side by side
- Read through conceptual pgvector and Pinecone code to see how the same workflow looks in a SQL database and in a fully managed cloud service, without needing to set either one up today

**Up next (Topic 5):** we'll put everything from this week together — chunking, embeddings, and a vector database — into one working semantic search engine, and compare it directly against naive keyword search.